In [30]:
from pathlib import Path
import os
import csv
from jsonargparse import CLI
import inspect

In [44]:
EPOCHS = 100
PATIENCE = 10

In [31]:
def ensure_readable(path: Path):
    if not path.exists() or not os.access(path, os.R_OK):
        raise FileNotFoundError(f'Cannot read file: {path}')

In [32]:
def resolve_image_path(csv_path: Path, filename: str) -> Path:
    image_path = Path(filename)
    if not image_path.is_absolute():
        image_path = (csv_path.parent / image_path).resolve()
    return image_path

In [33]:
def read_label_csv(csv_path: Path):
    ensure_readable(csv_path)
    with csv_path.open(newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None or 'filename' not in reader.fieldnames or 'class' not in reader.fieldnames:
            raise ValueError(f'CSV must contain filename and class columns: {csv_path}')
        rows = []
        for row in reader:
            if row.get('filename') is None or row.get('class') is None:
                continue
            rows.append({
                #'image_path': resolve_image_path(csv_path, row['filename']),
                'image_path': row['filename'],
                'class': int(row['class'])
            })
    return rows

In [ ]:
# needed AI to fix this due to WSL symlink issues
def make_link_or_copy(src: Path, dest: Path):
    src = src.resolve()
    try:
        os.symlink(src, dest)
    except FileExistsError:
        # dest exists (possibly a broken symlink); remove and retry
        try:
            dest.unlink()
            os.symlink(src, dest)
        except OSError:
            from shutil import copy2
            copy2(src, dest)
    except OSError:
        # symlinks may not be supported (e.g. some WSL/Windows filesystems)
        from shutil import copy2
        if dest.lexists() if hasattr(dest, 'lexists') else os.path.lexists(dest):
            dest.unlink(missing_ok=True)
        copy2(src, dest)        

In [35]:
def prepare_yolo_dataset(dataset_root: Path):
    func_name = inspect.currentframe().f_code.co_name
    print( f"** In {func_name}, tracing arguments..." )
    for name, value in locals().items():
        print(f'{name}: {value}')
    print('Tracing done.\n')
        
    train_csv = dataset_root / 'train' / 'train_images_and_labels.csv'
    val_csv = dataset_root / 'val' / 'val_images_and_labels.csv'

    train_rows = read_label_csv(train_csv)
    print( "first train row:", train_rows[0] )
    print( "last train row:", train_rows[-1] )          
    val_rows = read_label_csv(val_csv)
    print( "first validation row:", val_rows[0] )
    print( "last validation row:", val_rows[-1] )      

    label_map = {1: 0, 2: 1}
    class_names = ['day', 'night']
    yolo_root = dataset_root / 'yolo_cls'
    train_root = yolo_root / 'train'
    val_root = yolo_root / 'val'

    for subset_root in (train_root, val_root):
        for class_index in label_map.values():
            target_dir = (subset_root / str(class_index))
            print( f"Creating directory: {target_dir}" )
            target_dir.mkdir(parents=True, exist_ok=True)

    for rows, subset_root, csv_path in ((train_rows, train_root, train_csv), (val_rows, val_root, val_csv)):
        for idx, row in enumerate(rows):
            image_path = row['image_path']
            image_path_path = Path( image_path )
            
            if not image_path_path.exists():
                raise FileNotFoundError(f'Image not found: {image_path}')
            class_index = label_map.get(row['class'])
            if class_index is None:
                raise ValueError(f'Unsupported class label {row["class"]} in {csv_path}')
            dest = subset_root / str(class_index) / image_path_path.name
            if ( idx == 0 or idx == len(rows) - 1):
                print( f"Linking or copying image: {image_path_path} to {dest}" )
            make_link_or_copy(image_path_path, dest)

    data_yaml = yolo_root / 'data.yaml'
    data_yaml.write_text(
        f'train: {train_root}\n'
        f'val: {val_root}\n'
        f'nc: {len(class_names)}\n'
        f'names: {class_names}\n'
    )
    return data_yaml

In [43]:
def main(dataset: str = 'CODaN', model: str = 'yolov8n-cls',
         data_dir='../../data/CODaN',
         save_dir='../../models'):
    func_name = inspect.currentframe().f_code.co_name
    print( f"** In {func_name}, tracing arguments..." )
    for name, value in locals().items():
        print(f'{name}: {value}')
    print('Tracing done.\n')

    if dataset != 'CODaN':
        raise ValueError('YOLO training currently supports CODaN only.')

    dataset_root = data_dir
    dataset_root_path = Path(dataset_root)
    
    out_dir = save_dir + "/" + dataset + "/" + model
    out_dir_path = Path(out_dir)
    out_dir_path.mkdir(parents=True, exist_ok=True)
    print( f'Dataset directory for YOLO training: {dataset_root}')    
    print( f'Output directory for YOLO training: {out_dir}')

    data_yaml = prepare_yolo_dataset(dataset_root_path)
    data_yaml_parent = data_yaml.parent
    print(f'Prepared YOLO dataset config: {data_yaml_parent}')

    try:
        from ultralytics import YOLO
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            'ultralytics is required for YOLO training. Install it with `pip install ultralytics`.'
        ) from exc

    print('Starting YOLO training...')
    yolo_model = YOLO(model)
    results = yolo_model.train(
        data=str(data_yaml_parent),
        project=str(out_dir),
        name='train',
        exist_ok=True,
        workers=0, # needed for WSL
        epochs=EPOCHS,
        patience=PATIENCE
    )

    print('YOLO training complete.')
    print(f'Training outputs are available in: {out_dir}')
    return results

In [45]:
main()

** In main, tracing arguments...
dataset: CODaN
model: yolov8n-cls
data_dir: ../../data/CODaN
save_dir: ../../models
func_name: main
Tracing done.

Dataset directory for YOLO training: ../../data/CODaN
Output directory for YOLO training: ../../models/CODaN/yolov8n-cls
** In prepare_yolo_dataset, tracing arguments...
dataset_root: ../../data/CODaN
func_name: prepare_yolo_dataset
Tracing done.

first train row: {'image_path': '../../data/CODaN/train/night_Bus_2015_02014.jpg', 'class': 2}
last train row: {'image_path': '../../data/CODaN/train/night_Bottle_2015_01475.jpg', 'class': 2}
first validation row: {'image_path': '../../data/CODaN/val/night_Chair_2015_04288.jpg', 'class': 2}
last validation row: {'image_path': '../../data/CODaN/val/night_Bicycle_2015_00287.jpg', 'class': 2}
Creating directory: ../../data/CODaN/yolo_cls/train/0
Creating directory: ../../data/CODaN/yolo_cls/train/1
Creating directory: ../../data/CODaN/yolo_cls/val/0
Creating directory: ../../data/CODaN/yolo_cls/val/1

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b9ad312fcb0>
curves: []
curves_results: []
fitness: 0.949999988079071
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8999999761581421, 'metrics/accuracy_top5': 1.0, 'fitness': 0.949999988079071}
save_dir: PosixPath('/home/schelian/prjs/python_project_template/src/models/models/CODaN/yolov8n-cls/train')
speed: {'preprocess': 0.09369219915242866, 'inference': 0.409993200446479, 'loss': 0.00022084859665483236, 'postprocess': 0.000392099900636822}
top1: 0.8999999761581421
top5: 1.0